In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("DPA-Part1") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()


Read ListenBrainz to explore the schema

In [3]:
temp = spark.read.parquet(
    "../data/raw/mb/listenbrainz-spark-dump-2702-20261006-000002-incremental/",
    "../data/raw/mb/listenbrainz-spark-dump-2704-20261007-000002-incremental/",
    "../data/raw/mb/listenbrainz-spark-dump-2706-20261008-000002-incremental/"
)


Print the schema of the raw ListenBrainz data

In [4]:
temp.printSchema()

root
 |-- listened_at: timestamp (nullable = true)
 |-- created: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- recording_msid: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_credit_id: long (nullable = true)
 |-- release_name: string (nullable = true)
 |-- release_mbid: string (nullable = true)
 |-- recording_name: string (nullable = true)
 |-- recording_mbid: string (nullable = true)
 |-- artist_credit_mbids: array (nullable = true)
 |    |-- element: string (containsNull = true)



Load ListenBrainz with explicit schema

In [5]:
from pyspark.sql.types import *

lb_schema = StructType([
    StructField("listened_at", TimestampType()),
    StructField("created", TimestampType()),
    StructField("user_id", LongType()),
    StructField("recording_msid", StringType()),
    StructField("artist_name", StringType()),
    StructField("artist_credit_id", LongType()),
    StructField("release_name", StringType()),
    StructField("release_mbid", StringType()),
    StructField("recording_name", StringType()),
    StructField("recording_mbid", StringType()),
    StructField("artist_credit_mbids", ArrayType(StringType())),
])

listens = spark.read.schema(lb_schema).parquet(
    "../data/raw/mb/listenbrainz-spark-dump-2702-20261006-000002-incremental/*.parquet",
    "../data/raw/mb/listenbrainz-spark-dump-2704-20261007-000002-incremental/*.parquet",
    "../data/raw/mb/listenbrainz-spark-dump-2706-20261008-000002-incremental/*.parquet"
)

print("Total listens:", listens.count())


Total listens: 11459693


Preview the listens data

In [6]:
listens.show(3, truncate=True, vertical=True)

-RECORD 0-----------------------------------
 listened_at         | 2026-10-04 23:58:09  
 created             | 2026-10-05 00:00:... 
 user_id             | 188459               
 recording_msid      | a2677d64-37e6-40a... 
 artist_name         | Simba Musical        
 artist_credit_id    | 4405796              
 release_name        | Fiebre De Cumbia ... 
 release_mbid        | 00404921-2342-4ee... 
 recording_name      | Oasis De Amor        
 recording_mbid      | c8099998-484d-4fd... 
 artist_credit_mbids | [117149a1-9807-4f... 
-RECORD 1-----------------------------------
 listened_at         | 2026-10-04 23:59:55  
 created             | 2026-10-05 00:00:... 
 user_id             | 78821                
 recording_msid      | 2cd0a363-0c0b-4fa... 
 artist_name         | Zedd feat. Selena... 
 artist_credit_id    | 1438664              
 release_name        | True Colors          
 release_mbid        | d16f754f-e6cb-4c3... 
 recording_name      | I Want You to Know   
 recording

Peek at raw MusicBrainz artist file

In [7]:
raw_artist = spark.read.text("../data/raw/mb/mbdump/artist")
raw_artist.show(3, truncate=False, vertical=True)


-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------------------------------
 value | 2252039\tfadeb38c-833f-40bc-9d8c-a6383b38b1be\tДоктор Сатана\tДоктор Сатана\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\t0\t2021-11-23 07:08:52.479537+00\tf\t\N\t\N 
-RECORD 1---------------------------------------------------------------------------------------------------------------------------------------------------------------
 value | 371203\t49add228-eac5-4de8-836c-d75cde7369c3\tPete Moutso\tMoutso, Pete\t\N\t\N\t\N\t\N\t\N\t\N\t1\t\N\t\N\t\t0\t\N\tf\t\N\t\N                                 
-RECORD 2---------------------------------------------------------------------------------------------------------------------------------------------------------------
 value | 3087346\tdfdce491-133d-4e9f-9e48-795587e181b0\tUNlT\tUNlT\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\t0\t2025-09-14 23:53:28.004798+00\tf\t\N\t\N      

Load artist with explicit schema

In [8]:
artist_schema = StructType([
    StructField("id", IntegerType()),
    StructField("gid", StringType()),
    StructField("name", StringType()),
    StructField("sort_name", StringType()),
    StructField("begin_date_year", IntegerType()),
    StructField("begin_date_month", IntegerType()),
    StructField("begin_date_day", IntegerType()),
    StructField("end_date_year", IntegerType()),
    StructField("end_date_month", IntegerType()),
    StructField("end_date_day", IntegerType()),
    StructField("type", IntegerType()),
    StructField("area", IntegerType()),
    StructField("gender", IntegerType()),
    StructField("comment", StringType()),
    StructField("edits_pending", IntegerType()),
    StructField("last_updated", StringType()),
    StructField("ended", StringType()),
    StructField("begin_area", IntegerType()),
    StructField("end_area", IntegerType()),
])

artist = spark.read.schema(artist_schema) \
    .option("sep", "\t") \
    .option("nullValue", "\\N") \
    .csv("../data/raw/mb/mbdump/artist")

print("Total artists:", artist.count())
artist.show(3, vertical=True, truncate=True)


Total artists: 3015712
-RECORD 0--------------------------------
 id               | 2252039              
 gid              | fadeb38c-833f-40b... 
 name             | Доктор Сатана        
 sort_name        | Доктор Сатана        
 begin_date_year  | NULL                 
 begin_date_month | NULL                 
 begin_date_day   | NULL                 
 end_date_year    | NULL                 
 end_date_month   | NULL                 
 end_date_day     | NULL                 
 type             | NULL                 
 area             | NULL                 
 gender           | NULL                 
 comment          | NULL                 
 edits_pending    | 0                    
 last_updated     | 2021-11-23 07:08:... 
 ended            | f                    
 begin_area       | NULL                 
 end_area         | NULL                 
-RECORD 1--------------------------------
 id               | 371203               
 gid              | 49add228-eac5-4de... 
 name      

Peek at raw MusicBrainz area file

In [10]:
raw_area = spark.read.text("../data/raw/mb/mbdump/area")
raw_area.show(3, truncate=False, vertical=True)


-RECORD 0-----------------------------------------------------------------------------------------------------------------------
 value | 15449\t2913ad77-cec8-4d2f-98d3-d4aa46ab73bc\tGreccio\t4\t0\t2013-07-21 22:47:57.660809+00\t\N\t\N\t\N\t\N\t\N\t\N\tf\t 
-RECORD 1-----------------------------------------------------------------------------------------------------------------------
 value | 38\t71bbafaa-e825-3e15-8ca9-017dcad1748b\tCanada\t1\t0\t2013-05-27 13:15:52.179105+00\t\N\t\N\t\N\t\N\t\N\t\N\tf\t     
-RECORD 2-----------------------------------------------------------------------------------------------------------------------
 value | 43\t82d5f4d6-aed4-3ff5-81d1-5363ac6e97a7\tChile\t1\t0\t2013-05-27 12:52:17.320228+00\t\N\t\N\t\N\t\N\t\N\t\N\tf\t      
only showing top 3 rows



Load area with explicit schema

In [9]:
area_schema = StructType([
    StructField("id", IntegerType()),
    StructField("gid", StringType()),
    StructField("name", StringType()),
    StructField("type", IntegerType()),
    StructField("edits_pending", IntegerType()),
    StructField("last_updated", StringType()),
    StructField("begin_date_year", IntegerType()),
    StructField("begin_date_month", IntegerType()),
    StructField("begin_date_day", IntegerType()),
    StructField("end_date_year", IntegerType()),
    StructField("end_date_month", IntegerType()),
    StructField("end_date_day", IntegerType()),
    StructField("ended", StringType()),
    StructField("comment", StringType()),
])

area = spark.read.schema(area_schema) \
    .option("sep", "\t") \
    .option("nullValue", "\\N") \
    .csv("../data/raw/mb/mbdump/area")

print("Total areas:", area.count())
area.show(3, vertical=True, truncate=True)


Total areas: 120108
-RECORD 0--------------------------------
 id               | 15449                
 gid              | 2913ad77-cec8-4d2... 
 name             | Greccio              
 type             | 4                    
 edits_pending    | 0                    
 last_updated     | 2013-07-21 22:47:... 
 begin_date_year  | NULL                 
 begin_date_month | NULL                 
 begin_date_day   | NULL                 
 end_date_year    | NULL                 
 end_date_month   | NULL                 
 end_date_day     | NULL                 
 ended            | f                    
 comment          | NULL                 
-RECORD 1--------------------------------
 id               | 38                   
 gid              | 71bbafaa-e825-3e1... 
 name             | Canada               
 type             | 1                    
 edits_pending    | 0                    
 last_updated     | 2013-05-27 13:15:... 
 begin_date_year  | NULL                 
 begin_date_mo

Write to Parquet

In [12]:
listens.write.mode("overwrite").parquet("../data/processed/listens")
artist.write.mode("overwrite").parquet("../data/processed/artist")
area.write.mode("overwrite").parquet("../data/processed/area")

print("Parquet tables written successfully")


ConnectionRefusedError: [Errno 111] Connection refused

In [39]:

from pyspark.sql import functions as F

initial_count = listens.count()

# Remove extra spaces from artist names
listens_clean = listens.withColumn(
    "artist_name", F.trim("artist_name")
)

# Remove rows with missing essential fields
listens_clean = listens_clean.filter(
    F.col("listened_at").isNotNull() &
    F.col("user_id").isNotNull() &
    F.col("recording_msid").isNotNull() &
    F.col("artist_name").isNotNull() &
    (F.col("artist_name") != "")
)

missing_removed = initial_count - listens_clean.count()
print("Rows removed (missing values):", missing_removed)


NameError: name 'listens' is not defined

In [ ]:

before_dedup = listens_clean.count()

listens_clean = listens_clean.dropDuplicates()

duplicates_removed = before_dedup - listens_clean.count()
print("Duplicate rows removed:", duplicates_removed)


In [ ]:

before_checks = listens_clean.count()

listens_clean = listens_clean.filter(
    (F.col("user_id") >= 0) &
    (F.col("listened_at") <= F.current_timestamp()) &
    (
        F.col("created").isNull() |
        (F.col("created") <= F.current_timestamp())
    )
)

invalid_removed = before_checks - listens_clean.count()
print("Invalid rows removed:", invalid_removed)


In [ ]:

# Clean artist table
artist_clean = artist.withColumn("name", F.trim("name"))

artist_clean = artist_clean.filter(
    F.col("id").isNotNull() &
    F.col("gid").isNotNull() &
    F.col("name").isNotNull() &
    (F.col("name") != "")
).dropDuplicates()

# Clean area table
area_clean = area.withColumn("name", F.trim("name"))

area_clean = area_clean.filter(
    F.col("id").isNotNull() &
    F.col("gid").isNotNull() &
    F.col("name").isNotNull() &
    (F.col("name") != "")
).dropDuplicates()

print("Clean artists:", artist_clean.count())
print("Clean areas:", area_clean.count())


In [ ]:

final_count = listens_clean.count()

print("Initial listening rows:", initial_count)
print("Rows removed (missing values):", missing_removed)
print("Duplicate rows removed:", duplicates_removed)
print("Invalid rows removed:", invalid_removed)
print("Final listening rows:", final_count)


In [ ]:

listens_clean.write.mode("overwrite").parquet(
    "../data/processed/listens_clean"
)

artist_clean.write.mode("overwrite").parquet(
    "../data/processed/artist_clean"
)

area_clean.write.mode("overwrite").parquet(
    "../data/processed/area_clean"
)